# **Numerical Method in Finance Final Project**

# **1. Problem Description**
Implement a PDE pricer in Python which allows you to price the below derivatives. You may make the following assumptions



1. Vol is flat at 16%

2. Interest rate is 5% flat per annum

3. Time to expiration is measured in terms of fraction per annum, i.e. 0.5 is exactly half a year to maturity.

4. Assume zero dividend and borrow of stock.  Assume the growth rate is the same as the rate used in discounting.


Price the below derivatives:

1. European options, by entering call/put type, strike, time to maturity, spot price. Compare it with your Black-Scholes pricer.
2. American options, by entering call/put type, strike, time to maturity, spot price. Observe it should be priced higher than the corresponding European option.
3. A continuous up and out barrier option, by entering call/put type, time to maturity, barrier level(assume barrier level is greater than strike), strike, spot price.  Observe it should be priced cheaper than the corresponding European option

# **2. Mathematical Formulas**
## **Black-Scholes PDE**

$$
\frac{\partial V}{\partial t} + (r-q)S \frac{\partial V}{\partial S} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} = rV
$$
where $V(S,t)$ is option price, $S$ is underlying asset price, $\sigma$ is volatility, and $r$ is risk-free interest rate.

\\
For a call option, the terminal condition is defined by the payoff at maturity $T$:
$$
V(S, T) = \max(S - K, 0)
$$
where $K$ is strike price.

And the Dirichlet boundary conditions are:
$$
V(0, t) = 0
$$
    
$$
V(S_{max}, t) = S_{max} - Ke^{-r(T-t)}
$$

For a put option, the terminal condition is:
$$
V(S, T) = \max(K - S, 0)
$$
The Dirichlet boundary conditions are:
$$
V(0, t) = Ke^{-r(T-t)}
$$
    
$$
V(S_{max}, t) = 0
$$

\\
Neumann conditions specify the partial derivative of the option at the boundary. Discretizing the second derivative at the boundary by central difference gives us a more friendly equation about $V$. Additionally, the Neumann boundary conditions are inclined to be more accurate for the same boundaries than the Dirichlet condtions, as the second derivatives falls off faster than the price. And they are more universal, same for both call and put, and on both ends.
$$
\frac{\partial^2 V}{\partial S^2}=0
$$

## **Build the grid**
Discretize the space and time where $i\in\{0,1,2,\cdots,M\}$ denotes the space steps and $j\in\{0,1,2,\cdots,N\}$ denote the time steps, set $\Delta S=\frac{S_{max}}{M}$, $\Delta t = \frac{T}{N}$, then we have $S_i=i\cdot\Delta S$ and  $t_j=j\cdot\Delta t$.

## **Space discretization**
Using central difference,
$$
\frac{\partial V}{\partial S} \approx \frac{V_{i+1,j} - V_{i-1,j}}{2\Delta S}
$$

$$
\frac{\partial^2 V}{\partial S^2} \approx \frac{V_{i+1,j} - 2V_{i,j} + V_{i-1,j}}{\Delta S^2}
$$

And we can write Neumann boundary conditions in the following forms:
$$
V_{0,j} - 2V_{1,j} + V_{2,j}=0
$$

$$
V_{M,j} - 2V_{M-1,j} + V_{M-2,j}=0
$$

## **Time discretization**
**1.Explicit method**

Using backward difference,
$$
\frac{\partial V}{\partial t} \approx \frac{V_{i,j} - V_{i,j-1}}{\Delta t}
$$
Plugging them in the PDE equation, we get
$$
V_{i,j-1}=a_{i}V_{i-1,j}+(1+b_{i})V_{i,j}+c_{i}V_{i+1,j}
$$
where
$$
a_{i} = \frac{1}{2}i\Delta t(i\sigma^2-r+q)
$$

$$
b_{i} = -\Delta t(i^2\sigma^2+r)
$$

$$
c_{i} = \frac{1}{2}i\Delta t(i\sigma^2+r-q)
$$

**2.Implicit method**

Using forward difference,
$$
\frac{\partial V}{\partial t} \approx \frac{V_{i,j+1} - V_{i,j}}{\Delta t}
$$
Plugging them in the PDE equation, we get
$$
V_{i,j+1}=a_{i}V_{i-1,j}+(1-b_{i})V_{i,j}+c_{i}V_{i+1,j}
$$
where
$$
a_{i} = \frac{1}{2}i\Delta t(r-q-i\sigma^2)
$$

$$
b_{i} = -\Delta t(r+i^2\sigma^2)
$$

$$
c_{i} = \frac{1}{2}i\Delta t(r-q+i\sigma^2)
$$

**3.Crank-Nicolson**

It's the combination of Explicit and Implicit methods,
$$
\frac{\partial V}{\partial t} \approx \frac{V_{i,j+1} - V_{i,j}}{\Delta t}
$$

$$
\frac{\partial V}{\partial S} \approx \frac{1}{2}(\frac{V_{i+1,j+1} - V_{i-1,j+1}}{2\Delta S}+\frac{V_{i+1,j} - V_{i-1,j}}{2\Delta S})
$$

$$
\frac{\partial^2 V}{\partial S^2} \approx \frac{1}{2}(\frac{V_{i+1,j+1} - 2V_{i,j+1} + V_{i-1,j+1}}{\Delta S^2} + \frac{V_{i+1,j} - 2V_{i,j} + V_{i-1,j}}{\Delta S^2})
$$
Plugging them in the PDE equation, we get
$$
-a_{i}V_{i-1,j}+(1-b_{i})V_{i,j}-c_{i}V_{i+1,j}=a_{i}V_{i-1,j+1}+(1+b_{i})V_{i,j+1}+c_{i}V_{i+1,j+1}
$$
where
$$
a_{i} = \frac{1}{4}i\Delta t(i\sigma^2-r+q)
$$

$$
b_{i} = -\frac{1}{2}\Delta t(i^2\sigma^2+r)
$$

$$
c_{i} = \frac{1}{4}i\Delta t(i\sigma^2+r-q)
$$

## **Matrix form for European option**
Denote
$$
V_j=
\begin{bmatrix}
V_{1,j} \\
V_{2,j} \\
\vdots \\
V_{M-1,j} \\
\end{bmatrix}
$$

**1.Explicit method**

Rearragne the equation and plug in the Neumann boundary conditions, we get
$$
V_{j-1} = PV_j
$$
where
$$
P =
\begin{bmatrix}
1+b_1+2a_1 & c_1-a_1 & 0 & \cdots & 0 \\
a_2 & 1+b_2 & c_2 & \cdots & 0 \\
0 & a_3 & 1+b_3 & \cdots & 0 \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
0 & \ddots & a_{M-2} & 1+b_{M-2} & c_{M-2} \\
0 & \cdots & 0 & a_{M-1}-c_{M-1} & 1+b_{M-1}+2c_{M-1} \\
\end{bmatrix}
$$

**2.Implicit method**

Rearragne the equation and plug in the Neumann boundary conditions, we get
$$
V_{j+1} = QV_j
$$
where
$$
Q =
\begin{bmatrix}
1-b_1+2a_1 & c_1-a_1 & 0 & \cdots & 0 \\
a_2 & 1-b_2 & c_2 & \cdots & 0 \\
0 & a_3 & 1-b_3 & \cdots & 0 \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
0 & \ddots & a_{M-2} & 1-b_{M-2} & c_{M-2} \\
0 & \cdots & 0 & a_{M-1}-c_{M-1} & 1-b_{M-1}+2c_{M-1} \\
\end{bmatrix}
$$
Then, $V_{j} = Q^{-1}V_{j+1}$.

**3.Crank-Nicolson**

Rearragne the equation and plug in the Neumann boundary conditions, we get
$$
M_1V_{j+1} = M_2V_j
$$
where
$$
M1 =
\begin{bmatrix}
1+b_1+2a_1 & c_1-a_1 & 0 & \cdots & 0 \\
a_2 & 1+b_2 & c_2 & \cdots & 0 \\
0 & a_3 & 1+b_3 & \cdots & 0 \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
0 & \ddots & a_{M-2} & 1+b_{M-2} & c_{M-2} \\
0 & \cdots & 0 & a_{M-1}-c_{M-1} & 1+b_{M-1}+2c_{M-1} \\
\end{bmatrix}
$$
and
$$
M2 =
\begin{bmatrix}
1-b_1-2a_1 & -c_1+a_1 & 0 & \cdots & 0 \\
-a_2 & 1-b_2 & -c_2 & \cdots & 0 \\
0 & -a_3 & 1-b_3 & \cdots & 0 \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
0 & \ddots & -a_{M-2} & 1-b_{M-2} & c_{M-2} \\
0 & \cdots & 0 & -a_{M-1}+c_{M-1} & 1-b_{M-1}-2c_{M-1} \\
\end{bmatrix}
$$
Then, $V_{j} = M_2^{-1}M_1V_{j+1}$.

## **Other options**
**1. American option**

Denote the intrinsic value as $IV_{i,j}$, then for call option, $IV_{i,j}=S_{i,j}-K$ and for put option, $IV_{i,j}=K-S_{i,j}$.
At each time step and for each space step, we have to adjust the option value:
$$
V_{i,j} = max(V_{i,j}, IV_{i,j})
$$

**2. Continuous up and out barrier option**

Denote the barrier as $B$. At each time step and for each space step, if $S_{i,j}>B$, we have to adjust the option value:
$$
V_{i,j} = 0
$$

# **3. Code**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import time
from scipy.stats import norm
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

In [ ]:
class Finite_Diff_pricer():
  def __init__(self, S0, K, T, sigma, r, q, N, M, option_type, barrier=-1):
    self.S0 = S0  # Spot price of underlying asset
    self.K = K  # Strike price
    self.T = T  # Maturity
    self.sigma = sigma  # Volatility of underlying asset
    self.r = r  # Risk-free rate
    self.q = q  # Dividend yield
    self.M = M  # Space steps
    self.N = N  # Time steps
    self.option_type = option_type.lower()  # Indicator for "call" or "put" option
    self.barrier = barrier  # Barrier for option
    self.S_max = max(self.K * np.exp(3 * self.sigma * np.sqrt(self.T)), 2 * self.S0)  # Max state for grid
    self.dS = self.S_max / self.M  # Space step size
    self.dt = self.T / self.N  # Time step size
    self.iValues = np.arange(1, self.M - 1)
    self.jValues = np.arange(1, self.N - 1)
    self.SValues = np.linspace(0, self.S_max, self.M)
    self.V = np.zeros(shape=(self.M, self.N))  # grid of option value
    self.set_conditions()  # Initial and boundary conditions
    self.price = -1  # Numerical price
    self.bs_price = self.black_scholes()  # Analytical price

  def set_conditions(self):
    if self.option_type == "call":
      # Terminal condition
      self.V[:, -1] = np.maximum(self.SValues - self.K, 0)
      # Boundary conditions
      self.V[0, 1:-1] = 0
      self.V[-1, 1:-1] = self.S_max - self.K * np.exp(-self.r * (self.T - self.jValues * self.dt))
    elif self.option_type == "put":
      # Terminal condition
      self.V[:, -1] = np.maximum(self.K - self.SValues, 0)
      # Boundary conditions
      self.V[0, 1:-1] = self.K * np.exp(-self.r * (self.T - self.jValues * self.dt))
      self.V[-1, 1:-1] = 0
    else:
      raise ValueError("Invalid opiton type. Must be either 'call' or 'put'.")

  def black_scholes(self):
    """
    Black-Scholes price for European option.
    """
    d1 = (np.log(self.S0 / self.K) + (self.r + 0.5 * self.sigma**2) * self.T) / (self.sigma * np.sqrt(self.T))
    d2 = d1 - self.sigma*np.sqrt(self.T)
    if self.option_type == 'call':
        price = self.S0*norm.cdf(d1) - self.K*np.exp(-self.r*self.T)*norm.cdf(d2)
    else:
        price = self.K*np.exp(-self.r*self.T)*norm.cdf(-d2) - self.S0*norm.cdf(-d1)
    self.price = price
    return price

  def Explicit(self):
    size = self.M - 2
    a = 0.5 * self.iValues * self.dt * (self.iValues * self.sigma**2 - self.r + self.q)
    b = -self.dt * (self.iValues**2 * self.sigma**2 + self.r)
    c = 0.5 * self.iValues * self.dt * (self.iValues * self.sigma**2 + self.r - self.q)

    M = diags(diagonals=[a[1:], 1 + b, c[:-1]], offsets=[-1, 0, 1], shape=(size, size), format='lil')
    M[0, 0] += 2*a[0]
    M[0, 1] -= a[0]
    M[-1, -2] -= c[-1]
    M[-1, -1] += 2*c[-1]
    return M.tocsc()

  def Implicit(self):
    size = self.M - 2
    a = 0.5 * self.iValues * self.dt * (self.r - self.q - self.iValues * self.sigma**2)
    b = - self.dt * (self.r + self.iValues**2 * self.sigma**2)
    c = -0.5 * self.iValues * self.dt * (self.r - self.q + self.iValues * self.sigma**2)

    M = diags(diagonals=[a[1:], 1 - b, c[:-1]], offsets=[-1, 0, 1], shape=(size, size), format='lil')
    M[0, 0] += 2*a[0]
    M[0, 1] -= a[0]
    M[-1, -2] -= c[-1]
    M[-1, -1] += 2*c[-1]
    return M.tocsc()

  def Crank_Nicolson(self):
    size = self.M - 2
    a = 0.25 * self.iValues * self.dt * (self.iValues * self.sigma**2 - self.r + self.q)
    b = -0.5 * self.dt * (self.iValues**2 * self.sigma**2 + self.r)
    c = 0.25 * self.iValues * self.dt * (self.iValues * self.sigma**2 + self.r - self.q)

    M1 = diags(diagonals=[a[1:], 1 + b, c[:-1]], offsets=[-1, 0, 1], shape=(size, size), format='lil')
    M1[0, 0] += 2*a[0]
    M1[0, 1] -= a[0]
    M1[-1, -2] -= c[-1]
    M1[-1, -1] += 2*c[-1]

    M2 = diags(diagonals=[-a[1:], 1 - b, -c[:-1]], offsets=[-1, 0, 1], shape=(size, size), format='lil')
    M2[0, 0] -= 2*a[0]
    M2[0, 1] += a[0]
    M2[-1, -2] += c[-1]
    M2[-1, -1] -= 2*c[-1]

    return M1.tocsc(), M2.tocsc()

  def European(self, method):
    """
    Finite difference method for European option.
    """
    if method == "Explicit":
      M = self.Explicit()
      for j in range(self.N-2, -1, -1):
        self.V[1:-1, j] = M.dot(self.V[1:-1, j+1])
    elif method == "Implicit":
      M = self.Implicit()
      for j in range(self.N-2, -1, -1):
        self.V[1:-1, j] = spsolve(M, self.V[1:-1, j+1])
    elif method == "Crank-Nicolson":
      M1, M2 = self.Crank_Nicolson()
      for j in range(self.N-2, -1, -1):
        self.V[1:-1, j] = spsolve(M2, M1.dot(self.V[1:-1, j+1]))

    self.price =  np.interp(self.S0, self.SValues, self.V[:, 0])

  def American(self, method):
    """
    Finite difference method for American option.
    """
    intrinsic_value = np.maximum(self.SValues[1:-1] - self.K, 0) if self.option_type == 'call' else np.maximum(self.K - self.SValues[1:-1], 0)
    if method == "Explicit":
      M = self.Explicit()
      for j in range(self.N-2, -1, -1):
        self.V[1:-1, j] = np.maximum(M.dot(self.V[1:-1, j+1]), intrinsic_value)
    elif method == "Implicit":
      M = self.Implicit()
      for j in range(self.N-2, -1, -1):
        self.V[1:-1, j] = np.maximum(spsolve(M, self.V[1:-1, j+1]), intrinsic_value)
    elif method == "Crank-Nicolson":
      M1, M2 = self.Crank_Nicolson()
      for j in range(self.N-2, -1, -1):
        self.V[1:-1, j] = np.maximum(spsolve(M2, M1.dot(self.V[1:-1, j+1])), intrinsic_value)

    self.price =  np.interp(self.S0, self.SValues, self.V[:, 0])

  def UO_cont_Barrier(self, method):
    """
    Finite difference method for continuous up and out Barrier option.
    """
    if method == "Explicit":
      M = self.Explicit()
      for j in range(self.N-2, -1, -1):
        self.V[1:-1, j] = np.where(self.SValues[1:-1] < self.barrier, M.dot(self.V[1:-1, j+1]), 0)
    elif method == "Implicit":
      M = self.Implicit()
      for j in range(self.N-2, -1, -1):
        self.V[1:-1, j] = np.where(self.SValues[1:-1] < self.barrier, spsolve(M, self.V[1:-1, j+1]), 0)
    elif method == "Crank-Nicolson":
      M1, M2 = self.Crank_Nicolson()
      for j in range(self.N-2, -1, -1):
        self.V[1:-1, j] = np.where(self.SValues[1:-1] < self.barrier, spsolve(M2, M1.dot(self.V[1:-1, j+1])), 0)

    self.price =  np.interp(self.S0, self.SValues, self.V[:, 0])

In [ ]:
# Parameters
S0 = 150  # spot price
K = 150  # strike price
T = 0.5  # maturity
sigma = 0.16  # volatility
r = 0.05  # risk-free rate
q = 0  # dividend rate
N = 100  # time steps
M = 100  # state steps
barrier =  160 # Barrier price

opt = Finite_Diff_pricer(S0, K, T, sigma, r, q, N, M, "put", barrier)

# **4. Numerical tests Results**

## **(1) CN PDE**

In [ ]:
rows_CN = []
for i in range(1, 16):
    M = i * 100
    N = i * 100

    opt_p_CN = Finite_Diff_pricer(S0, K, T, sigma, r, q, N, M, "put", barrier)
    opt_p_CN.European("Crank-Nicolson")
    eu_put = opt_p_CN.price
    opt_p_CN.American("Crank-Nicolson")
    am_put = opt_p_CN.price
    opt_p_CN.UO_cont_Barrier("Crank-Nicolson")
    uo_put = opt_p_CN.price
    opt_p_CN.black_scholes()
    bs_put = opt_p_CN.price
    opt_c_CN = Finite_Diff_pricer(S0, K, T, sigma, r, q, N, M, "call", barrier)
    opt_c_CN.European("Crank-Nicolson")
    eu_call = opt_c_CN.price
    opt_c_CN.American("Crank-Nicolson")
    am_call = opt_c_CN.price
    opt_c_CN.UO_cont_Barrier("Crank-Nicolson")
    uo_call = opt_c_CN.price
    opt_c_CN.black_scholes()
    bs_call = opt_c_CN.price

    rows_CN.append({
            "i": i, "M": M, "N": N,
            "eu_call": eu_call, "bs_call": bs_call,
            "am_call": am_call, "am>eu(call)": am_call > eu_call,
            "uo_call": uo_call, "uo<eu(call)": uo_call < eu_call,
            "eu_put": eu_put, "bs_put": bs_put,
            "am_put": am_put, "am>eu(put)": am_put > eu_put,
            "uo_put": uo_put, "uo<eu(put)": uo_put < eu_put
        })

df_CN = pd.DataFrame(rows_CN)
pd.set_option('display.float_format', '{:.6f}'.format)
df_CN

,i,M,N,eu_call,bs_call,am_call,am>eu(call),uo_call,uo<eu(call),eu_put,bs_put,am_put,am>eu(put),uo_put,uo<eu(put)
0,1,100,100,8.668878,8.697206,8.668878,False,0.283615,True,5.001943,4.993692,5.349324,True,4.291756,True
1,2,200,200,8.676719,8.697206,8.676719,False,0.340497,True,4.991494,4.993692,5.347124,True,4.384933,True
2,3,300,300,8.682171,8.697206,8.682171,False,0.264833,True,4.990850,4.993692,5.348832,True,4.266407,True
3,4,400,400,8.685417,8.697206,8.685417,False,0.231100,True,4.991048,4.993692,5.350060,True,4.199993,True
4,5,500,500,8.687530,8.697206,8.687530,False,0.259523,True,4.991332,4.993692,5.350916,True,4.260649,True
5,6,600,600,8.689007,8.697206,8.689007,False,0.237496,True,4.991589,4.993692,5.351546,True,4.217115,True
6,7,700,700,8.690095,8.697206,8.690095,False,0.222436,True,4.991807,4.993692,5.352016,True,4.184519,True
7,8,800,800,8.690929,8.697206,8.690929,False,0.240352,True,4.991988,4.993692,5.352378,True,4.224867,True
8,9,900,900,8.691589,8.697206,8.691589,False,0.228060,True,4.992140,4.993692,5.352674,True,4.198692,True
9,10,1000,1000,8.692124,8.697206,8.692124,False,0.218485,True,4.992268,4.993692,5.352912,True,4.177126,True


### **Analysis of the Crank Nicolso PDE Pricing Results**
1. European vs Black-Scholes

* The European Call values and Black-Scholes values are very close

2. American Call vs European Call
* For American calls on non-dividend-paying stocks, early exercise is never optimal

3. Up-and-Out Call vs European Call
* An up-and-out barrier call becomes worthless if the barrier is hit. This additional knockout condition reduces the option's value.

4. American Put vs European Put
* Regardless of dividends, American puts can benefit from early exercise, especially when the underlying price drops far below the strike. This gives them extra value over European puts.

5. Up-and-Out Put vs European Put
* Similar to the call case — a barrier option carries the risk of early knockout, reducing its expected payoff compared to the European put.


## **(2) Implicit PDE**

In [ ]:
rows_implicit = []
for i in range(1, 16):
    M = i * 100
    N = i * 100

    opt_p_implicit = Finite_Diff_pricer(S0, K, T, sigma, r, q, N, M, "put", barrier)
    opt_p_implicit.European("Implicit")
    eu_put = opt_p_implicit.price
    opt_p_implicit.American("Implicit")
    am_put = opt_p_implicit.price
    opt_p_implicit.UO_cont_Barrier("Implicit")
    uo_put = opt_p_implicit.price
    opt_p_implicit.black_scholes()
    bs_put = opt_p_implicit.price
    opt_c_implicit = Finite_Diff_pricer(S0, K, T, sigma, r, q, N, M, "call", barrier)
    opt_c_implicit.European("Implicit")
    eu_call = opt_c_implicit.price
    opt_c_implicit.American("Implicit")
    am_call = opt_c_implicit.price
    opt_c_implicit.UO_cont_Barrier("Implicit")
    uo_call = opt_c_implicit.price
    opt_c_implicit.black_scholes()
    bs_call = opt_c_implicit.price

    rows_implicit.append({
            "i": i, "M": M, "N": N,
            "eu_call": eu_call, "bs_call": bs_call,
            "am_call": am_call, "am>eu(call)": am_call > eu_call,
            "uo_call": uo_call, "uo<eu(call)": uo_call < eu_call,
            "eu_put": eu_put, "bs_put": bs_put,
            "am_put": am_put, "am>eu(put)": am_put > eu_put,
            "uo_put": uo_put, "uo<eu(put)": uo_put < eu_put
        })

df_implicit = pd.DataFrame(rows_implicit)
pd.set_option('display.float_format', '{:.6f}'.format)
df_implicit

,i,M,N,eu_call,bs_call,am_call,am>eu(call),uo_call,uo<eu(call),eu_put,bs_put,am_put,am>eu(put),uo_put,uo<eu(put)
0,1,100,100,8.660491,8.697206,8.660491,False,0.301832,True,4.994009,4.993692,5.335458,True,4.318062,True
1,2,200,200,8.672499,8.697206,8.672499,False,0.359023,True,4.987502,4.993692,5.339677,True,4.406786,True
2,3,300,300,8.679357,8.697206,8.679357,False,0.278878,True,4.988187,4.993692,5.343672,True,4.290702,True
3,4,400,400,8.683306,8.697206,8.683306,False,0.242822,True,4.989051,4.993692,5.346134,True,4.224476,True
4,5,500,500,8.685841,8.697206,8.685841,False,0.271325,True,4.989734,4.993692,5.347740,True,4.281982,True
5,6,600,600,8.687599,8.697206,8.687599,False,0.247903,True,4.990258,4.993692,5.348869,True,4.238295,True
6,7,700,700,8.688889,8.697206,8.688889,False,0.231829,True,4.990666,4.993692,5.349706,True,4.205339,True
7,8,800,800,8.689874,8.697206,8.689874,False,0.249764,True,4.990990,4.993692,5.350346,True,4.243840,True
8,9,900,900,8.690651,8.697206,8.690651,False,0.236723,True,4.991252,4.993692,5.350858,True,4.217383,True
9,10,1000,1000,8.691279,8.697206,8.691279,False,0.226545,True,4.991469,4.993692,5.351271,True,4.195492,True


### **Analysis of the implicit PDE Pricing Results**
1. European vs Black-Scholes
*  European values and Black-Scholes value are very close.

2. American Call vs European Call
*  In a no-dividend setting, American calls should equal European calls, since early exercise offers no benefit.

3. Up-and-Out Call vs European Call
* An up-and-out barrier call becomes worthless if the barrier is hit. This additional knockout condition reduces the option’s value.

4. American Put vs European Put
* Regardless of dividends, American puts can benefit from early exercise, especially when the underlying price drops far below the strike. This gives them extra value over European puts.

5. Up-and-Out Put vs European Put
* Similar to the call case — a barrier option carries the risk of early knockout, reducing its expected payoff compared to the European put.

## **(3) Explicit Finite Difference**

Notice that here we set the relationship between Space Steps (M) and Time Steps (N) in Explicit Finite Difference Method following the below logic:

In the explicit finite difference method, numerical **stability** is a major concern. The time step$\Delta t$   must be **sufficiently small** relative to the space step $\Delta S$ to prevent the solution from becoming unstable or diverging.

To ensure stability, a commonly used **CFL condition** (Courant-Friedrichs-Lewy) is:

$$
\Delta t \leq \frac{(\Delta S)^2}{4 \sigma^2 S_{\text{max}}^2}
$$

Where:

- $ \Delta S = \frac{S_{\text{max}}}{M} $
- $ S_{\text{max}} $ is the maximum stock price considered on the spatial grid (often set as $ \max(2S_0, B) $, where $ B $ is a barrier level if applicable)
- $ \sigma $ is the volatility

Substituting $ \Delta S $ into the CFL condition, the recommended time step becomes:

$$
\Delta t = \frac{(S_{\text{max}} / M)^2}{4 \sigma^2 S_{\text{max}}^2} = \frac{1}{4 \sigma^2 M^2}
$$

To cover the full option lifetime \( T \), the number of time steps is:

$$
N = \left\lceil \frac{T}{\Delta t} \right\rceil = \left\lceil 4 \sigma^2 M^2 T \right\rceil
$$


In [ ]:
import math
rows_explicit = []
for i in range(1, 16):
    M = i * 100

    S_max = max(2*S0, barrier)
    dS = S_max / M
    dt = dS**2 / (4 * sigma**2 * S_max**2)
    N = math.ceil(T / dt)

    opt_p_explicit = Finite_Diff_pricer(S0, K, T, sigma, r, q, N, M, "put", barrier)
    opt_p_explicit.European("Explicit")
    eu_put = opt_p_explicit.price
    opt_p_explicit.American("Explicit")
    am_put = opt_p_explicit.price
    opt_p_explicit.UO_cont_Barrier("Explicit")
    uo_put = opt_p_explicit.price
    opt_p_explicit.black_scholes()
    bs_put = opt_p_explicit.price
    opt_c_explicit = Finite_Diff_pricer(S0, K, T, sigma, r, q, N, M, "call", barrier)
    opt_c_explicit.European("Explicit")
    eu_call = opt_c_explicit.price
    opt_c_explicit.American("Explicit")
    am_call = opt_c_explicit.price
    opt_c_explicit.UO_cont_Barrier("Explicit")
    uo_call = opt_c_explicit.price
    opt_c_explicit.black_scholes()
    bs_call = opt_c_explicit.price

    rows_explicit.append({
            "i": i, "M": M, "N": N,
            "eu_call": eu_call, "bs_call": bs_call,
            "am_call": am_call, "am>eu(call)": am_call > eu_call,
            "uo_call": uo_call, "uo<eu(call)": uo_call < eu_call,
            "eu_put": eu_put, "bs_put": bs_put,
            "am_put": am_put, "am>eu(put)": am_put > eu_put,
            "uo_put": uo_put, "uo<eu(put)": uo_put < eu_put
        })

df_explicit = pd.DataFrame(rows_explicit)
pd.set_option('display.float_format', '{:.6f}'.format)
df_explicit

,i,M,N,eu_call,bs_call,am_call,am>eu(call),uo_call,uo<eu(call),eu_put,bs_put,am_put,am>eu(put),uo_put,uo<eu(put)
0,1,100,512,8.713296,8.697206,8.713296,False,0.245508,True,5.016838,4.993692,5.371532,True,4.256125,True
1,2,200,2048,8.701153,8.697206,8.701153,False,0.306848,True,4.999404,4.993692,5.358936,True,4.354034,True
2,3,300,4608,8.698950,8.697206,8.698950,False,0.239971,True,4.996221,4.993692,5.356839,True,4.231172,True
3,4,400,8192,8.698184,8.697206,8.698184,False,0.210484,True,4.995112,4.993692,5.356130,True,4.163465,True
4,5,500,12800,8.697831,8.697206,8.697831,False,0.238855,True,4.994600,4.993692,5.355807,True,4.227774,True
5,6,600,18432,8.697639,8.697206,8.697639,False,0.219169,True,4.994322,4.993692,5.355634,True,4.183793,True
6,7,700,25089,8.697524,8.697206,8.697524,False,0.205786,True,4.994155,4.993692,5.355531,True,4.151158,True
7,8,800,32768,8.697449,8.697206,8.697449,False,0.223592,True,4.994046,4.993692,5.355464,True,4.193855,True
8,9,900,41472,8.697398,8.697206,8.697398,False,0.212520,True,4.993972,4.993692,5.355418,True,4.167709,True
9,10,1000,51200,8.697361,8.697206,8.697361,False,0.203928,True,4.993919,4.993692,5.355386,True,4.146296,True


### **Analysis of the Explicit PDE Pricing Results**
1. European vs Black-Scholes
*  European values is very close to Black-Scholes values

2. American Call vs European Call
*  In a no-dividend setting, American calls should equal European calls, since early exercise offers no benefit.

3. Up-and-Out Call vs European Call
* An up-and-out barrier call becomes worthless if the barrier is hit. This additional knockout condition reduces the option’s value.

4. American Put vs European Put
* Regardless of dividends, American puts can benefit from early exercise, especially when the underlying price drops far below the strike. This gives them extra value over European puts.

5. Up-and-Out Put vs European Put
* Similar to the call case — a barrier option carries the risk of early knockout, reducing its expected payoff compared to the European put.

# **5. Summary**

All three finite difference methods — Crank-Nicolson, Implicit, and Explicit — produced option prices that align well with theoretical expectations. Across all methods, the European option prices closely matched the Black-Scholes analytical values, confirming the numerical accuracy. As expected, American call options equaled European calls in the no-dividend setting, while American puts consistently showed higher values due to the early exercise premium. Up-and-out barrier options, both calls and puts, were correctly priced lower than their European counterparts, reflecting the impact of the knockout condition.

In terms of stability, the Explicit method requires a small time step (CFL condition) to ensure numerical stability, making it conditionally stable. In contrast, both the Implicit and Crank-Nicolson methods are unconditionally stable, allowing for larger time steps without compromising accuracy. Among the three, Crank-Nicolson offers the best balance between stability and second-order accuracy in both time and space.